# Lab tracking — một lần Save & Run All

**Lưu Quang Khải · MSSV 2A202602599**

Import notebook này, gắn Input chứa `kaggle_upload.zip` và dữ liệu lab, bật **GPU + Internet**,
rồi **Save Version → Save & Run All**. Xem `HUONG_DAN_KAGGLE.md` trước khi chạy.
Notebook tự phát hiện đường dẫn khi Input có đúng một bản repo và một bộ lab.

Luồng chạy đủ frame cho ba cấu hình/video, chấm **chỉ video_1**, xuất TXT của mọi
cấu hình và minh chứng. Bốn video không nhãn cần xem để chốt lựa chọn; báo cáo nháp
chưa tự tạo quan sát. Tải `ket_qua_2A202602599.zip` ở Output và gửi lại để hoàn thiện báo cáo.


In [1]:
from pathlib import Path
import os
import sys
import subprocess
import shutil
import zipfile

STUDENT = "Lưu Quang Khải"
STUDENT_ID = "2A202602599"
REPO_HINT = "/kaggle/input/datasets/luuquangkhai/labday21v3"  # Đường dẫn repo đã giải nén; để trống để tự tìm.
LAB_DATA_HINT = "/kaggle/input/datasets/luuquangkhai/labday22/data_lab21/data_lab21"  # Thư mục chứa video_1…video_5; để trống để tự tìm.
TRACKEVAL_COMMIT = "12c8791b303e0a0b50f753af204249e622d0281a"

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
TEMP = Path("/kaggle/temp/tracking_lab")
TEMP.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
if not INPUT.exists():
    raise RuntimeError("Notebook này cần chạy trên Kaggle và có Dataset trong Input.")
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
# BoxMOT cũ cần cách nạp checkpoint Re-ID cố định theo hành vi torch.load trước đây.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"


## Tìm mã nguồn và dữ liệu

Chỉ giải nén ZIP có cấu trúc repo hoặc lab; không tải ảnh lên nguồn công khai.

In [2]:
extracted = TEMP / "inputs"
extracted.mkdir(exist_ok=True)
direct_repos = [path for path in INPUT.rglob("run_kaggle_lab.py") if path.parent.name == "scripts"]
direct_labs = [path for path in INPUT.rglob("video_1")
               if path.is_dir() and all((path.parent / f"video_{i}" / "img1").is_dir()
                                       for i in range(1, 6))]
for number, archive in enumerate(sorted(INPUT.rglob("*.zip"))):
    with zipfile.ZipFile(archive) as source:
        names = [item.filename.replace("\\", "/") for item in source.infolist()]
        is_repo = not direct_repos and any(name.endswith("scripts/run_kaggle_lab.py") for name in names)
        is_lab = not direct_labs and any("video_1/img1/" in name for name in names)
        if not (is_repo or is_lab):
            continue
        destination = extracted / f"goi_{number}"
        destination.mkdir(exist_ok=True)
        for name in names:
            target = (destination / name).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise RuntimeError(f"ZIP có đường dẫn không hợp lệ: {archive}")
        source.extractall(destination)

roots = [INPUT, extracted]
repos = sorted({path.parents[1] for root in roots for path in root.rglob("run_kaggle_lab.py")
                if path.parent.name == "scripts"})
labs = sorted({path.parent for root in roots for path in root.rglob("video_1")
               if path.is_dir() and all((path.parent / f"video_{i}" / "img1").is_dir()
                                       for i in range(1, 6))})
if REPO_HINT:
    repos = [Path(REPO_HINT)]
if LAB_DATA_HINT:
    labs = [Path(LAB_DATA_HINT)]
if len(repos) != 1 or len(labs) != 1:
    raise RuntimeError(f"Cần đúng một repo và một bộ lab. Repo: {repos}; lab: {labs}. "
                       "Kiểm tra Add Input, bỏ bản trùng hoặc điền đường dẫn ở ô đầu.")
REPO = TEMP / "repo"
shutil.copytree(repos[0], REPO, dirs_exist_ok=True)
LAB_DATA = labs[0]
os.environ["LAB_DATA"] = str(LAB_DATA)
os.chdir(REPO)
print("Repo:", REPO, "\nDữ liệu:", LAB_DATA)


Repo: /kaggle/temp/tracking_lab/repo 
Dữ liệu: /kaggle/input/datasets/luuquangkhai/labday22/data_lab21/data_lab21


## Cài Python 3.11 riêng, PyTorch GPU và TrackEval

Kernel Kaggle có thể là Python 3.13. Luồng lab dùng Python 3.11 riêng do uv tải về, không dùng thư viện Python 3.13 của kernel. Cần Internet để tải Python và PyTorch CUDA.

In [3]:
print("Python của kernel Kaggle:", sys.version)
subprocess.run(["nvidia-smi", "-L"], check=True)
UV_TOOLS = TEMP / "uv_tools"
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade",
                "--target", str(UV_TOOLS), "uv<1"], check=True)
uv_env = dict(os.environ)
uv_env["PYTHONPATH"] = str(UV_TOOLS)
uv_env["UV_PYTHON_INSTALL_DIR"] = str(TEMP / "pythons")
uv_env["UV_CACHE_DIR"] = str(TEMP / "uv_cache")
uv_command = [sys.executable, "-m", "uv"]
subprocess.run(uv_command + ["python", "install", "3.11"], check=True, env=uv_env)
VENV = TEMP / "venv_py311"
subprocess.run(uv_command + ["venv", "--python", "3.11", "--seed", "--allow-existing",
                              str(VENV)], check=True, env=uv_env)
PYTHON = VENV / "bin/python"
subprocess.run([str(PYTHON), "--version"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "torch==2.6.0", "torchvision==0.21.0",
                "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", str(REPO / "requirements-kaggle.txt")], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-deps", "boxmot==10.0.42"], check=True)
TRACKEVAL = TEMP / "TrackEval"
if not TRACKEVAL.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/JonathonLuiten/TrackEval.git",
                    str(TRACKEVAL)], check=True)
subprocess.run(["git", "-C", str(TRACKEVAL), "fetch", "--depth", "1", "origin", TRACKEVAL_COMMIT], check=True)
subprocess.run(["git", "-C", str(TRACKEVAL), "checkout", TRACKEVAL_COMMIT], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-deps", "-e", str(TRACKEVAL)], check=True)
if not shutil.which("ffmpeg"):
    raise RuntimeError("Image Kaggle thiếu ffmpeg. Chọn image có ffmpeg trước khi chạy luồng này.")


Python của kernel Kaggle: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
GPU 0: Tesla T4 (UUID: GPU-9ddd82be-11d0-317c-0b21-07abd30335ef)
GPU 1: Tesla T4 (UUID: GPU-e37113e8-5afc-b1d0-a54f-62497d6ce231)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 80.1 MB/s eta 0:00:00


 Downloaded cpython-3.11.16-linux-x86_64-gnu (download)
Installed Python 3.11.16 in 1.68s
 + cpython-3.11.16-linux-x86_64-gnu (python3.11)
Using CPython 3.11.16
Creating virtual environment with seed packages at: /kaggle/temp/tracking_lab/venv_py311
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source /kaggle/temp/tracking_lab/venv_py311/bin/activate


Python 3.11.16
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 129.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 402.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 146.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 50.3 MB/s  0:00:06
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 141.5 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 160.3 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 155.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 175.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 152.4 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 160.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 165.7 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Cloning into '/kaggle/temp/tracking_lab/TrackEval'...
From https://github.com/JonathonLuiten/TrackEval
 * branch            12c8791b303e0a0b50f753af204249e622d0281a -> FETCH_HEAD
Note: switching to '12c8791b303e0a0b50f753af204249e622d0281a'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 12c8791 added tabulate requirements and caught import error in init (#102)


Obtaining file:///kaggle/temp/tracking_lab/TrackEval
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for trackeval (pyproject.toml): started
  Building editable for trackeval (pyproject.toml): finished with status 'done'
  Created wheel for trackeval: filename=trackeval-1.0.dev1-0.editable-py3-none-any.whl size=8631 sha256=f740fc13fbfd44e00bcd0ca77e6d29e54202c1fea2aa70ced78d05e863d71f86
  Stored in directory: /tmp/pip-ephem-wheel-cache-sdgt3m2s/wheels/83/fe/e8/ef20a9dc8a971ed7ba913f134fe93ff903c460b151ab75c9e0
Successfully bui

## Kiểm tra toàn bộ luồng trên mẫu nhỏ — 20 frame/video

Chạy cả 15 lượt trên bản mẫu riêng, kiểm tra cache, preview/ffmpeg, đánh giá video_1 và đóng ZIP. Chỉ chạy toàn bộ khi bước này đạt. Số liệu mẫu không dùng cho báo cáo chính.

In [4]:
subprocess.run([str(PYTHON), "-c",
    "import torch, numpy, cv2, ultralytics, boxmot; "
    "assert torch.cuda.is_available(), 'Chưa bật GPU trong Settings'; "
    "print('Thử tính toán CUDA:', (torch.ones(2, device='cuda') * 2).sum().item()); "
    "print('GPU:', torch.cuda.get_device_name(0)); "
    "print('NumPy:', numpy.__version__, 'YOLO:', ultralytics.__version__, 'BoxMOT:', boxmot.__version__)"], check=True)
subprocess.run([str(PYTHON), "-m", "pytest", "-q"], check=True)
subprocess.run([str(PYTHON), "scripts/check_data.py", "--lab-data-root", str(LAB_DATA)], check=True)
subprocess.run([str(PYTHON), "-c",
    "from pathlib import Path; import gdown; "
    "from boxmot.appearance.reid_model_factory import get_model_url; "
    "weights = Path('osnet_x0_25_msmt17.pt'); "
    "weights.is_file() or gdown.download(get_model_url(weights), str(weights), quiet=False); "
    "assert weights.is_file() and weights.stat().st_size > 1024, 'Không tải được trọng số Re-ID cố định'"], check=True)
SMOKE_ROOT = TEMP / "kiem_tra_mau"
subprocess.run([str(PYTHON), "scripts/smoke_test_lab.py",
    "--lab-data-root", str(LAB_DATA), "--trackeval-root", str(TRACKEVAL),
    "--work-root", str(SMOKE_ROOT), "--frames", "20", "--device", "cuda:0"], check=True)
assert (SMOKE_ROOT / "DAT_KIEM_TRA.json").is_file(), "Mẫu chưa đạt; không chạy dữ liệu đầy đủ"


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


/kaggle/temp/tracking_lab/venv_py311/lib/python3.11/site-packages/boxmot/utils/checks.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Thử tính toán CUDA: 4.0
GPU: Tesla T4
NumPy: 1.26.4 YOLO: 8.4.174 BoxMOT: 10.0.42
.........................................                                [100%]
41 passed in 0.21s
[OK   ] video_1    600 ảnh  có nhãn  — Quảng trường ngoài trời, camera TĨNH, ban ngày, mật độ trung bình — video luyện, có nhãn
[OK   ] video_2   1050 ảnh  không có nhãn (đúng)  — Phố, camera TĨNH trên cao, BAN ĐÊM, mật độ RẤT đông
[OK   ] video_3    837 ảnh  không có nhãn (đúng)  — Camera DI CHUYỂN, độ phân giải thấp, khung hình chậm
[OK   ] video_4    900 ảnh  không có nhãn (đúng)  — TRONG NHÀ, camera di chuyển tiến về phía trước, phản chiếu kính
[OK   ] video_5    750 ảnh  không có nhãn (đúng)  — Quay từ XE BUS tại giao lộ đông, rung lắc mạnh

Preview .mp4: 0 file trong /kaggle/input/datasets/luuquangkhai/labday22/data_lab21/data_lab21/preview
Gói dữ liệu đủ để chạy scripts/run_tracking.py.


/kaggle/temp/tracking_lab/venv_py311/lib/python3.11/site-packages/boxmot/utils/checks.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Downloading...
From: https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF
To: /kaggle/temp/tracking_lab/repo/osnet_x0_25_msmt17.pt
100%|██████████| 3.06M/3.06M [00:00<00:00, 206MB/s]


[OK   ] video_1    600 ảnh  có nhãn  — Quảng trường ngoài trời, camera TĨNH, ban ngày, mật độ trung bình — video luyện, có nhãn
[OK   ] video_2   1050 ảnh  không có nhãn (đúng)  — Phố, camera TĨNH trên cao, BAN ĐÊM, mật độ RẤT đông
[OK   ] video_3    837 ảnh  không có nhãn (đúng)  — Camera DI CHUYỂN, độ phân giải thấp, khung hình chậm
[OK   ] video_4    900 ảnh  không có nhãn (đúng)  — TRONG NHÀ, camera di chuyển tiến về phía trước, phản chiếu kính
[OK   ] video_5    750 ảnh  không có nhãn (đúng)  — Quay từ XE BUS tại giao lộ đông, rung lắc mạnh

Preview .mp4: 0 file trong /kaggle/input/datasets/luuquangkhai/labday22/data_lab21/data_lab21/preview
Gói dữ liệu đủ để chạy scripts/run_tracking.py.
Không có eval_config.json; dùng tên thư mục chấm LAB/train cho video_1.
[OK   ] video_1     20 ảnh  có nhãn  — Quảng trường ngoài trời, camera TĨNH, ban ngày, mật độ trung bình — video luyện, có nhãn
[OK   ] video_2     20 ảnh  không có nhãn (đúng)  — Phố, camera TĨNH trên cao, BAN ĐÊM, mật độ RẤ

## Chạy đầy đủ và đóng gói

Không giới hạn frame trong bước này. Luồng sẽ dừng nếu bất kỳ lượt chạy hoặc đánh giá nào lỗi.

In [5]:
assert (SMOKE_ROOT / "DAT_KIEM_TRA.json").is_file(), "Cần chạy cell kiểm tra mẫu trước"
OUTPUT = WORK / "ket_qua_lab"
subprocess.run([str(PYTHON), "scripts/run_kaggle_lab.py",
    "--lab-data-root", str(LAB_DATA), "--trackeval-root", str(TRACKEVAL),
    "--out", str(OUTPUT), "--cache-root", str(TEMP / "cache_day_du"),
    "--device", "cuda:0", "--student", STUDENT, "--student-id", STUDENT_ID], check=True)


[OK   ] video_1    600 ảnh  có nhãn  — Quảng trường ngoài trời, camera TĨNH, ban ngày, mật độ trung bình — video luyện, có nhãn
[OK   ] video_2   1050 ảnh  không có nhãn (đúng)  — Phố, camera TĨNH trên cao, BAN ĐÊM, mật độ RẤT đông
[OK   ] video_3    837 ảnh  không có nhãn (đúng)  — Camera DI CHUYỂN, độ phân giải thấp, khung hình chậm
[OK   ] video_4    900 ảnh  không có nhãn (đúng)  — TRONG NHÀ, camera di chuyển tiến về phía trước, phản chiếu kính
[OK   ] video_5    750 ảnh  không có nhãn (đúng)  — Quay từ XE BUS tại giao lộ đông, rung lắc mạnh

Preview .mp4: 0 file trong /kaggle/input/datasets/luuquangkhai/labday22/data_lab21/data_lab21/preview
Gói dữ liệu đủ để chạy scripts/run_tracking.py.
Không có eval_config.json; dùng tên thư mục chấm LAB/train cho video_1.
absl-py==2.5.0
anyio==4.15.1
beautifulsoup4==4.15.0
boxmot==10.0.42
certifi==2026.7.22
charset-normalizer==3.5.2
cloudpickle==3.1.2
contourpy==1.3.3
cycler==0.12.1
filelock==3.32.3
filterpy==1.4.5
fonttools==4.66.1
fsspec==20

CompletedProcess(args=['/kaggle/temp/tracking_lab/venv_py311/bin/python', 'scripts/run_kaggle_lab.py', '--lab-data-root', '/kaggle/input/datasets/luuquangkhai/labday22/data_lab21/data_lab21', '--trackeval-root', '/kaggle/temp/tracking_lab/TrackEval', '--out', '/kaggle/working/ket_qua_lab', '--cache-root', '/kaggle/temp/tracking_lab/cache_day_du', '--device', 'cuda:0', '--student', 'Lưu Quang Khải', '--student-id', '2A202602599'], returncode=0)

## Tải kết quả

Dùng Output của phiên Save & Run All đã lưu. Gửi ZIP để xem minh chứng và hoàn thiện báo cáo; kiểm tra lựa chọn tạm thời trước khi nộp.

In [6]:
import json
from IPython.display import FileLink, display

manifest = json.loads((OUTPUT / "manifest.json").read_text(encoding="utf-8"))
assert manifest["status"] == "complete"
archive = WORK / f"ket_qua_{STUDENT_ID}.zip"
assert archive.exists()
print(f"Đã chạy đủ {len(manifest['records'])} lượt. Gói tải về: {archive}")
print(f"Dung lượng ZIP: {archive.stat().st_size / 1024**2:.1f} MB")
print("Bốn video không nhãn đang tạm chọn BoT-SORT; cần xem minh chứng để chốt.")
os.chdir(WORK)
display(FileLink(archive.name))
print((OUTPUT / "BAO_CAO_nhap.md").read_text(encoding="utf-8"))


Đã chạy đủ 15 lượt. Gói tải về: /kaggle/working/ket_qua_2A202602599.zip
Dung lượng ZIP: 59.4 MB
Bốn video không nhãn đang tạm chọn BoT-SORT; cần xem minh chứng để chốt.


/kaggle/working/ket_qua_2A202602599.zip

# Bản nháp báo cáo lab tracking

**Sinh viên:** Lưu Quang Khải · **MSSV:** 2A202602599

Detector cố định: `yolo26n.pt`, ảnh 640 px, lớp người; Re-ID `osnet_x0_25_msmt17.pt`.

Bản nháp tự điền cấu hình và số liệu thực tế. Cần xem minh chứng để hoàn thiện quan sát và xác nhận lựa chọn cho video_2–video_5 trước khi nộp.

## 1. Cấu hình

| Video | Cấu hình | Tracker | conf | iou | Cách chọn |
|---|---|---|---|---|---|
| video_1 | botsort_c030_i050 | botsort | 0.3 | 0.5 | HOTA cao nhất trong ba cấu hình; hòa điểm xét IDF1 rồi MOTA |
| video_2 | botsort_c030_i050 | botsort | 0.3 | 0.5 | Tạm chọn; cần xem minh chứng để xác nhận hoặc đổi cấu hình |
| video_3 | botsort_c030_i050 | botsort | 0.3 | 0.5 | Tạm chọn; cần xem minh chứng để xác nhận hoặc đổi cấu hình |
| video_4 | botsort_c030_i050 | botsort | 0.3 | 0.5 | Tạm chọn; cần xem minh chứng để xác nhận hoặc đổi cấu hình |
| video_5 | botsort_c030_i050 | botsort | 0.3 | 0.5 | Tạm chọn; cần xem minh chứng để xác nhận hoặc đổi cấu hình |

## 2.